In [1]:
import pandas as pd 
import numpy as np 
import MetaTrader5 as mt5
from datetime import datetime, timezone
from pathlib import Path

# Pipeline configurations

In [2]:
# Assets configuration

assets = ['EURUSDm','USDCADm','USDJPYm','GBPUSDm','USDCHFm','EURCADm',
          'EURCHFm','EURGBPm','EURJPYm','UKOILm','USOILm','XAUUSDm', 'XAGUSDm']

In [3]:
# Candle timeframe

timeframe = mt5.TIMEFRAME_D1

In [4]:
# Defining initial date to get data

initial_date = datetime(2010, 1, 1, tzinfo=timezone.utc)

# Defining final date to finishing proccess

final_date = datetime.now(timezone.utc)

In [5]:
# Storage and get data function

def data_storage(asset):
    
    # Search for candles on timeframe defined

    rates = mt5.copy_rates_range(asset,
                                 timeframe,
                                 initial_date,
                                 final_date)
    
    # If have a error, break the proccess

    if rates is None or len(rates) == 0:
        print(f'Fail to get data asset {asset}: {mt5.last_error()}')
        return None
    
    # Convert get data in dataframe

    df = pd.DataFrame(rates)

    # Convert the timestamps Unix to datetime with timezone UTC

    df['time'] = pd.to_datetime(df['time'], unit='s', utc=True)

    # Order olders candles to new candles

    df = df.sort_values('time')

    # Remove evenctually register with duplicate timestamp

    df = df.drop_duplicates(subset='time', keep='last')

    # Verify if essential values have a data

    essential_colums = ['time','open','high','low',
                        'close']
    
    if df[essential_colums].isna().any().any():
        raise ValueError(f'{asset}: there is NaN values presents on dataframe')
    
    # Verify if asset price are positive

    if (df[['open','high','low','close']] <= 0).any().any():
        raise ValueError(f'{asset}: there is negative price on dataframe')
    
    # Remove last candle, because most be in close processing

    if not df.empty:
        df = df.iloc[:-1].copy()

    # Adds the asset name to identify the origin of data

    df['symbol'] = asset 

    # Return validated dataframe

    return df

# Create a pipeline to extract Macro indicators

In [6]:
# Read macro indicators of Euro zone, USA, Canada, Switzerland, England and Japan central banks

canada = pd.read_csv(r'C:\\Christian_meus_estudos\\financial_market_quantitative_analyses\\pipeline_market\\storage\\canada_macro.csv',
                     sep=',', encoding='utf8')
euro_zone = pd.read_csv(r'C:\\Christian_meus_estudos\\financial_market_quantitative_analyses\\pipeline_market\\storage\\euro_macro.csv',
                        sep=',', encoding='utf8')
japan = pd.read_csv(r'C:\\Christian_meus_estudos\\financial_market_quantitative_analyses\\pipeline_market\\storage\\japan_macro.csv',
                    sep=',', encoding='utf8')
switz = pd.read_csv(r'C:\\Christian_meus_estudos\\financial_market_quantitative_analyses\\pipeline_market\\storage\\switzerland_macro.csv',
                    sep=',', encoding='utf8')
england = pd.read_csv(r'C:\\Christian_meus_estudos\\financial_market_quantitative_analyses\\pipeline_market\\storage\\uk_macro.csv',
                    sep=',', encoding='utf8')
usa = pd.read_csv(r'C:\\Christian_meus_estudos\\financial_market_quantitative_analyses\\pipeline_market\\storage\\usa_macro.csv',
                    sep=',', encoding='utf8')

In [7]:
canada

,reference_date,inflation_yoy,central_bank_rate,real_gdp_growth_qoq
0,1990-01-01,5.502063,NaN,NaN
1,1990-02-01,5.464481,NaN,NaN
2,1990-03-01,5.298913,NaN,0.961690
3,1990-04-01,5.013550,NaN,NaN
4,1990-05-01,4.423592,NaN,NaN
...,...,...,...,...
436,2026-05-01,3.225806,2.25,NaN
437,2026-06-01,2.798054,2.25,0.819738
438,2026-07-01,3.032141,2.25,NaN
439,2026-08-01,3.033981,2.25,NaN


In [8]:
euro_zone

,reference_date,inflation_yoy,central_bank_rate,real_gdp_growth_qoq,ecb_deposit_rate
0,1990-01-01,NaN,NaN,NaN,NaN
1,1990-02-01,NaN,NaN,NaN,NaN
2,1990-03-01,NaN,NaN,NaN,NaN
3,1990-04-01,NaN,NaN,NaN,NaN
4,1990-05-01,NaN,NaN,NaN,NaN
...,...,...,...,...,...
6372,2026-10-16,NaN,NaN,NaN,2.5
6373,2026-10-17,NaN,NaN,NaN,2.5
6374,2026-10-18,NaN,NaN,NaN,2.5
6375,2026-10-19,NaN,NaN,NaN,2.5


In [9]:
japan

,reference_date,inflation_yoy,central_bank_rate,real_gdp_growth_qoq
0,1990-01-01,3.280840,4.25,NaN
1,1990-02-01,3.820817,4.25,NaN
2,1990-03-01,3.669724,5.25,NaN
3,1990-04-01,2.574002,5.25,NaN
4,1990-05-01,2.560820,5.25,NaN
...,...,...,...,...
436,2026-05-01,1.500000,0.75,NaN
437,2026-06-01,1.701701,1.00,0.360456
438,2026-07-01,1.898101,1.00,NaN
439,2026-08-01,NaN,1.00,NaN


In [10]:
switz

,reference_date,inflation_yoy,central_bank_rate,real_gdp_growth_qoq
0,1990-01-01,NaN,NaN,NaN
1,1990-02-01,NaN,NaN,NaN
2,1990-03-01,NaN,NaN,NaN
3,1990-04-01,NaN,NaN,NaN
4,1990-05-01,NaN,NaN,NaN
...,...,...,...,...
436,2026-05-01,0.879648,0.0,NaN
437,2026-06-01,0.727962,0.0,1.884751
438,2026-07-01,0.675810,0.0,NaN
439,2026-08-01,0.915332,0.0,NaN


In [11]:
england

,reference_date,inflation_yoy,central_bank_rate,real_gdp_growth_qoq
0,1990-01-01,5.511811,14.875,NaN
1,1990-02-01,5.882353,14.875,NaN
2,1990-03-01,5.859375,14.875,0.514596
3,1990-04-01,6.358382,14.875,NaN
4,1990-05-01,6.704981,14.875,NaN
...,...,...,...,...
436,2026-05-01,2.890173,3.750,NaN
437,2026-06-01,2.591793,3.750,0.532607
438,2026-07-01,2.805755,3.750,NaN
439,2026-08-01,3.086863,3.750,NaN


In [12]:
usa 

,reference_date,cpi_index,fed_funds_rate,real_gdp
0,2010-01-01,217.488,0.11,16582.710
1,2010-02-01,217.281,0.13,NaN
2,2010-03-01,217.353,0.16,NaN
3,2010-04-01,217.403,0.20,16743.162
4,2010-05-01,217.290,0.20,NaN
...,...,...,...,...
196,2026-05-01,333.979,3.63,NaN
197,2026-06-01,332.568,3.63,NaN
198,2026-07-01,332.813,3.63,NaN
199,2026-08-01,334.131,3.63,NaN


# Data frame processing and data preparation for macroeconomic analysis.

In [13]:
canada

,reference_date,inflation_yoy,central_bank_rate,real_gdp_growth_qoq
0,1990-01-01,5.502063,NaN,NaN
1,1990-02-01,5.464481,NaN,NaN
2,1990-03-01,5.298913,NaN,0.961690
3,1990-04-01,5.013550,NaN,NaN
4,1990-05-01,4.423592,NaN,NaN
...,...,...,...,...
436,2026-05-01,3.225806,2.25,NaN
437,2026-06-01,2.798054,2.25,0.819738
438,2026-07-01,3.032141,2.25,NaN
439,2026-08-01,3.033981,2.25,NaN


In [14]:
"""
    Here, replacing the data with the median is neither the best nor the recommended approach, 
    but since extracting macroeconomic data generally requires an API key from reliable paid sites like Bloomberg,
    And at the moment I don't have such resources for a portfolio project, 
    I decided to extract the data directly from the official central bank websites of each country in question 
    and replace it with the median values ​​to simplify the project 
    and prepare the model for financial market analysis and forecasting, more specifically, the forex market.
"""

def prepare_df_macro(df):
    
    # Copy df

    df = df.copy()

    # Replacing NaN values ​​with the median

    for i in df.columns:
        if i == 'reference_date':
            continue

        df[i] = df[i].fillna(df[i].median())

    # Return df

    return df

In [15]:
canada.isna().sum()

reference_date           0
inflation_yoy            1
central_bank_rate      231
real_gdp_growth_qoq    295
dtype: int64

In [16]:
canada = prepare_df_macro(canada)

In [17]:
canada

,reference_date,inflation_yoy,central_bank_rate,real_gdp_growth_qoq
0,1990-01-01,5.502063,1.00,0.636564
1,1990-02-01,5.464481,1.00,0.636564
2,1990-03-01,5.298913,1.00,0.961690
3,1990-04-01,5.013550,1.00,0.636564
4,1990-05-01,4.423592,1.00,0.636564
...,...,...,...,...
436,2026-05-01,3.225806,2.25,0.636564
437,2026-06-01,2.798054,2.25,0.819738
438,2026-07-01,3.032141,2.25,0.636564
439,2026-08-01,3.033981,2.25,0.636564


In [18]:
canada.isna().sum()

reference_date         0
inflation_yoy          0
central_bank_rate      0
real_gdp_growth_qoq    0
dtype: int64

In [19]:
euro_zone = prepare_df_macro(euro_zone)
japan = prepare_df_macro(japan)
switz = prepare_df_macro(switz)
england = prepare_df_macro(england)
usa = prepare_df_macro(usa)

# A very important observation here:

In a real investment environment, for data that is not obtained and comes in NaN format, the best approach would be to exclude the NaN rows or seek another data source.

However, as I mentioned, in this market, the best data sources are paid, and for a portfolio project where I handle everything from the initial pipeline process, data extraction, and preparation, there is no need to invest financially in such a project or prepare a mega-structure for something that I may not even execute in the financial market investing real money; it's just study and learning.

But I'm just justifying the practice I adopted of replacing NaN values ​​with the median to facilitate the development of econometric models, because the problem with consulting central bank APIs is that many series are discontinued, many data points are not filled in correctly, generating gaps, etc.

The best way to decide between filling in with the mean, median, or deleting those NaN rows depends on your business rules and how sensitive your model will be to the data.

In [20]:
canada

,reference_date,inflation_yoy,central_bank_rate,real_gdp_growth_qoq
0,1990-01-01,5.502063,1.00,0.636564
1,1990-02-01,5.464481,1.00,0.636564
2,1990-03-01,5.298913,1.00,0.961690
3,1990-04-01,5.013550,1.00,0.636564
4,1990-05-01,4.423592,1.00,0.636564
...,...,...,...,...
436,2026-05-01,3.225806,2.25,0.636564
437,2026-06-01,2.798054,2.25,0.819738
438,2026-07-01,3.032141,2.25,0.636564
439,2026-08-01,3.033981,2.25,0.636564


# Returning to extracting data from the financial market with MT5

In [21]:
# Executing pipeline MT5

def main():

    # Initialize MetaTrader5 terminal connection

    if not mt5.initialize():
        raise RuntimeError(
            f'Not possible connect to MT5: {mt5.last_error()}'
        )
    
    try:

        # Show informations terminal connected

        terminal = mt5.terminal_info()
        print('MetaTrader5 connected!')
        print(f'Connected terminal: {terminal is not None}')

        # Storage the dataframe colected

        dataframes = {}

        # It goes through all connected assets

        for asset in assets:
            print(f'Extracting data to {asset}')

            # Guarantees if selected asset be on terminal

            if not mt5.symbol_select(asset, True):
                print(f'Asset currently unavailable: {asset}')
                continue

            try: 

                # Executing strorage and mkae a dataframe

                df = data_storage(asset)

                # Ignore the asset if the collection did not return any data.

                if df is None or df.empty:
                    print(f'No close candle available: {asset}')
                    continue 

                # Save dataframe

                globals()[asset] = df

                # Show summary of storage

                print(f'Rows storaged: {len(df)}')
                print(f"First candle: {df['time'].min()}")
                print(f"Last candle {df['time'].max()}")
                

            except Exception as erro:

                # Fail register and keep goin with another assets

                print(f'Error to proccess {asset}: {erro}')

        # Show how many assets are processed

        print(f'Pipeline finish. Assets processed: {len(dataframes)}')

        # Return df

        return dataframes

    finally:

        # Closing MetaTrader5 connection even with error

        mt5.shutdown()
        print(f'\n MetaTrader5 connection closed')

In [22]:
# The MT5 pipeline only runs if the file is started.

if __name__ == '__main__':
    dataframes = main()

MetaTrader5 connected!
Connected terminal: True
Extracting data to EURUSDm
Rows storaged: 3942
First candle: 2014-01-14 00:00:00+00:00
Last candle 2026-10-19 00:00:00+00:00
Extracting data to USDCADm
Rows storaged: 1557
First candle: 2021-10-27 00:00:00+00:00
Last candle 2026-10-19 00:00:00+00:00
Extracting data to USDJPYm
Rows storaged: 1557
First candle: 2021-10-27 00:00:00+00:00
Last candle 2026-10-19 00:00:00+00:00
Extracting data to GBPUSDm
Rows storaged: 3942
First candle: 2014-01-14 00:00:00+00:00
Last candle 2026-10-19 00:00:00+00:00
Extracting data to USDCHFm
Rows storaged: 1557
First candle: 2021-10-27 00:00:00+00:00
Last candle 2026-10-19 00:00:00+00:00
Extracting data to EURCADm
Rows storaged: 1557
First candle: 2021-10-27 00:00:00+00:00
Last candle 2026-10-19 00:00:00+00:00
Extracting data to EURCHFm
Rows storaged: 1557
First candle: 2021-10-27 00:00:00+00:00
Last candle 2026-10-19 00:00:00+00:00
Extracting data to EURGBPm
Rows storaged: 1557
First candle: 2021-10-27 00:00

In [23]:
eurusd = EURUSDm

In [24]:
eurusd

,time,open,high,low,close,tick_volume,spread,real_volume,symbol
0,2014-01-14 00:00:00+00:00,1.36669,1.36983,1.36485,1.36661,58248,0,0,EURUSDm
1,2014-01-15 00:00:00+00:00,1.36665,1.36723,1.35807,1.35986,60319,0,0,EURUSDm
2,2014-01-16 00:00:00+00:00,1.35990,1.36487,1.35827,1.36145,72629,0,0,EURUSDm
3,2014-01-17 00:00:00+00:00,1.36129,1.36205,1.35166,1.35382,64588,0,0,EURUSDm
4,2014-01-19 00:00:00+00:00,1.35365,1.35396,1.35238,1.35264,1682,0,0,EURUSDm
...,...,...,...,...,...,...,...,...,...
3937,2026-10-14 00:00:00+00:00,1.12007,1.12111,1.11125,1.11940,38596,8,0,EURUSDm
3938,2026-10-15 00:00:00+00:00,1.11938,1.12407,1.11839,1.12122,47926,8,0,EURUSDm
3939,2026-10-16 00:00:00+00:00,1.12124,1.12692,1.12077,1.12388,32842,8,0,EURUSDm
3940,2026-10-18 00:00:00+00:00,1.11892,1.12303,1.11773,1.12260,2842,8,0,EURUSDm


In [25]:
eurcad = EURCADm
eurchf = EURCHFm
eurgbp = EURGBPm
eurjpy = EURJPYm
usdcad = USDCADm
usdchf = USDCHFm
usdjpy = USDJPYm
wti = USOILm
brent = UKOILm
gold = XAUUSDm
gbpusd = GBPUSDm
silver = XAGUSDm

In [26]:
eurcad

,time,open,high,low,close,tick_volume,spread,real_volume,symbol
0,2021-10-27 00:00:00+00:00,1.43125,1.43267,1.43125,1.43235,425,20,0,EURCADm
1,2021-10-28 00:00:00+00:00,1.43339,1.44342,1.43088,1.44238,23728,20,0,EURCADm
2,2021-10-29 00:00:00+00:00,1.44240,1.44415,1.42951,1.43132,30462,20,0,EURCADm
3,2021-10-31 00:00:00+00:00,1.43096,1.43191,1.43069,1.43102,1187,20,0,EURCADm
4,2021-11-01 00:00:00+00:00,1.43099,1.43616,1.42959,1.43527,22533,20,0,EURCADm
...,...,...,...,...,...,...,...,...,...
1552,2026-10-14 00:00:00+00:00,1.59238,1.59250,1.58612,1.59086,27806,15,0,EURCADm
1553,2026-10-15 00:00:00+00:00,1.59084,1.59608,1.58933,1.59141,33480,15,0,EURCADm
1554,2026-10-16 00:00:00+00:00,1.59140,1.59456,1.58791,1.58886,24066,15,0,EURCADm
1555,2026-10-18 00:00:00+00:00,1.57895,1.58538,1.57841,1.58493,1773,15,0,EURCADm


In [27]:
eurchf

,time,open,high,low,close,tick_volume,spread,real_volume,symbol
0,2021-10-27 00:00:00+00:00,1.06492,1.06492,1.06476,1.06492,70,29,0,EURCHFm
1,2021-10-28 00:00:00+00:00,1.06443,1.06698,1.06350,1.06497,9207,29,0,EURCHFm
2,2021-10-29 00:00:00+00:00,1.06499,1.06542,1.05687,1.05812,10830,29,0,EURCHFm
3,2021-10-31 00:00:00+00:00,1.05726,1.05883,1.05722,1.05863,832,29,0,EURCHFm
4,2021-11-01 00:00:00+00:00,1.05864,1.06028,1.05339,1.05473,9813,29,0,EURCHFm
...,...,...,...,...,...,...,...,...,...
1552,2026-10-14 00:00:00+00:00,0.93268,0.93341,0.93039,0.93103,13154,16,0,EURCHFm
1553,2026-10-15 00:00:00+00:00,0.93105,0.93145,0.92692,0.92809,17408,16,0,EURCHFm
1554,2026-10-16 00:00:00+00:00,0.92806,0.92890,0.92484,0.92603,14085,16,0,EURCHFm
1555,2026-10-18 00:00:00+00:00,0.92239,0.92493,0.92193,0.92470,985,16,0,EURCHFm


In [28]:
eurgbp

,time,open,high,low,close,tick_volume,spread,real_volume,symbol
0,2021-10-27 00:00:00+00:00,0.84414,0.84421,0.84406,0.84414,115,16,0,EURGBPm
1,2021-10-28 00:00:00+00:00,0.84274,0.84751,0.84221,0.84656,10785,16,0,EURGBPm
2,2021-10-29 00:00:00+00:00,0.84657,0.84720,0.84325,0.84428,14942,16,0,EURGBPm
3,2021-10-31 00:00:00+00:00,0.84431,0.84483,0.84416,0.84452,635,16,0,EURGBPm
4,2021-11-01 00:00:00+00:00,0.84454,0.84987,0.84417,0.84940,12577,16,0,EURGBPm
...,...,...,...,...,...,...,...,...,...
1552,2026-10-14 00:00:00+00:00,0.84484,0.84622,0.84262,0.84610,9601,13,0,EURGBPm
1553,2026-10-15 00:00:00+00:00,0.84608,0.84771,0.84455,0.84485,11127,13,0,EURGBPm
1554,2026-10-16 00:00:00+00:00,0.84487,0.84500,0.84182,0.84315,8409,13,0,EURGBPm
1555,2026-10-18 00:00:00+00:00,0.84017,0.84186,0.83932,0.84177,909,13,0,EURGBPm


In [29]:
eurjpy

,time,open,high,low,close,tick_volume,spread,real_volume,symbol
0,2021-10-27 00:00:00+00:00,132.056,132.084,132.018,132.084,237,19,0,EURJPYm
1,2021-10-28 00:00:00+00:00,131.692,132.726,131.549,132.721,20494,19,0,EURJPYm
2,2021-10-29 00:00:00+00:00,132.723,132.909,131.599,131.808,28448,19,0,EURJPYm
3,2021-10-31 00:00:00+00:00,131.801,132.047,131.797,132.009,1575,19,0,EURJPYm
4,2021-11-01 00:00:00+00:00,132.011,132.436,131.788,132.336,23391,19,0,EURJPYm
...,...,...,...,...,...,...,...,...,...
1552,2026-10-14 00:00:00+00:00,177.331,177.586,176.285,176.846,40469,16,0,EURJPYm
1553,2026-10-15 00:00:00+00:00,176.843,177.411,176.634,177.305,50785,16,0,EURJPYm
1554,2026-10-16 00:00:00+00:00,177.306,177.958,177.138,177.541,35813,16,0,EURJPYm
1555,2026-10-18 00:00:00+00:00,176.704,177.242,176.634,177.148,2758,16,0,EURJPYm


In [30]:
usdcad

,time,open,high,low,close,tick_volume,spread,real_volume,symbol
0,2021-10-27 00:00:00+00:00,1.23239,1.23416,1.23239,1.23370,530,12,0,USDCADm
1,2021-10-28 00:00:00+00:00,1.23681,1.23727,1.23293,1.23465,23003,12,0,USDCADm
2,2021-10-29 00:00:00+00:00,1.23469,1.24075,1.23270,1.23798,33752,12,0,USDCADm
3,2021-10-31 00:00:00+00:00,1.23795,1.23856,1.23759,1.23828,605,22,0,USDCADm
4,2021-11-01 00:00:00+00:00,1.23833,1.24029,1.23510,1.23732,23179,12,0,USDCADm
...,...,...,...,...,...,...,...,...,...
1552,2026-10-14 00:00:00+00:00,1.42167,1.42992,1.41964,1.42128,20085,14,0,USDCADm
1553,2026-10-15 00:00:00+00:00,1.42125,1.42215,1.41689,1.41935,22131,14,0,USDCADm
1554,2026-10-16 00:00:00+00:00,1.41936,1.41982,1.41338,1.41377,16597,14,0,USDCADm
1555,2026-10-18 00:00:00+00:00,1.41150,1.41322,1.41079,1.41184,1564,14,0,USDCADm


In [31]:
usdchf

,time,open,high,low,close,tick_volume,spread,real_volume,symbol
0,2021-10-27 00:00:00+00:00,0.91702,0.91739,0.91699,0.91728,239,10,0,USDCHFm
1,2021-10-28 00:00:00+00:00,0.91857,0.91942,0.91119,0.91168,22939,10,0,USDCHFm
2,2021-10-29 00:00:00+00:00,0.91169,0.91757,0.91018,0.91532,26238,10,0,USDCHFm
3,2021-10-31 00:00:00+00:00,0.91541,0.91628,0.91528,0.91612,454,15,0,USDCHFm
4,2021-11-01 00:00:00+00:00,0.91614,0.91710,0.90841,0.90933,20001,10,0,USDCHFm
...,...,...,...,...,...,...,...,...,...
1552,2026-10-14 00:00:00+00:00,0.83267,0.83943,0.83156,0.83169,20693,13,0,USDCHFm
1553,2026-10-15 00:00:00+00:00,0.83167,0.83238,0.82642,0.82774,32046,13,0,USDCHFm
1554,2026-10-16 00:00:00+00:00,0.82771,0.82817,0.82175,0.82392,24218,13,0,USDCHFm
1555,2026-10-18 00:00:00+00:00,0.82281,0.82504,0.82281,0.82370,1494,13,0,USDCHFm


In [32]:
usdjpy

,time,open,high,low,close,tick_volume,spread,real_volume,symbol
0,2021-10-27 00:00:00+00:00,113.710,113.769,113.697,113.767,326,8,0,USDJPYm
1,2021-10-28 00:00:00+00:00,113.638,113.717,113.251,113.611,22194,8,0,USDJPYm
2,2021-10-29 00:00:00+00:00,113.613,114.092,113.389,113.978,32390,8,0,USDJPYm
3,2021-10-31 00:00:00+00:00,114.015,114.250,114.003,114.232,1135,11,0,USDJPYm
4,2021-11-01 00:00:00+00:00,114.235,114.440,113.911,114.088,26075,8,0,USDJPYm
...,...,...,...,...,...,...,...,...,...
1552,2026-10-14 00:00:00+00:00,158.314,158.946,157.876,157.977,50733,10,0,USDJPYm
1553,2026-10-15 00:00:00+00:00,157.975,158.242,157.543,158.135,72505,10,0,USDJPYm
1554,2026-10-16 00:00:00+00:00,158.132,158.336,157.689,157.979,48237,10,0,USDJPYm
1555,2026-10-18 00:00:00+00:00,157.917,158.056,157.777,157.792,3746,10,0,USDJPYm


In [33]:
brent

,time,open,high,low,close,tick_volume,spread,real_volume,symbol
0,2021-10-27 00:00:00+00:00,83.883,83.883,83.581,83.676,534,96,0,UKOILm
1,2021-10-28 00:00:00+00:00,83.059,83.825,81.906,83.771,18350,92,0,UKOILm
2,2021-10-29 00:00:00+00:00,83.491,84.059,82.374,83.480,25574,94,0,UKOILm
3,2021-11-01 00:00:00+00:00,83.110,84.824,82.826,84.205,22001,93,0,UKOILm
4,2021-11-02 00:00:00+00:00,84.194,84.862,83.435,83.685,23644,93,0,UKOILm
...,...,...,...,...,...,...,...,...,...
1280,2026-10-13 00:00:00+00:00,104.054,105.136,102.649,104.785,120995,13,0,UKOILm
1281,2026-10-14 00:00:00+00:00,104.569,105.443,103.702,104.502,127960,13,0,UKOILm
1282,2026-10-15 00:00:00+00:00,103.955,104.662,101.949,104.050,148803,13,0,UKOILm
1283,2026-10-16 00:00:00+00:00,103.673,107.902,103.072,107.612,149772,13,0,UKOILm


In [34]:
silver

,time,open,high,low,close,tick_volume,spread,real_volume,symbol
0,2021-10-27 00:00:00+00:00,24.100,24.112,24.085,24.087,157,54,0,XAGUSDm
1,2021-10-28 00:00:00+00:00,24.084,24.220,23.936,24.039,9774,54,0,XAGUSDm
2,2021-10-29 00:00:00+00:00,24.037,24.054,23.633,23.862,15256,54,0,XAGUSDm
3,2021-10-31 00:00:00+00:00,23.829,23.832,23.722,23.750,416,54,0,XAGUSDm
4,2021-11-01 00:00:00+00:00,23.752,24.071,23.724,23.996,11764,54,0,XAGUSDm
...,...,...,...,...,...,...,...,...,...
1544,2026-10-14 00:00:00+00:00,58.979,59.363,55.380,55.716,83890,25,0,XAGUSDm
1545,2026-10-15 00:00:00+00:00,55.718,57.521,55.687,57.078,68296,25,0,XAGUSDm
1546,2026-10-16 00:00:00+00:00,57.073,57.970,56.874,56.975,63580,25,0,XAGUSDm
1547,2026-10-18 00:00:00+00:00,57.121,57.439,57.099,57.372,2783,25,0,XAGUSDm


In [35]:
gold

,time,open,high,low,close,tick_volume,spread,real_volume,symbol
0,2021-10-27 00:00:00+00:00,1795.864,1797.361,1795.695,1796.984,1275,200,0,XAUUSDm
1,2021-10-28 00:00:00+00:00,1799.738,1810.380,1792.306,1799.575,76894,200,0,XAUUSDm
2,2021-10-29 00:00:00+00:00,1799.559,1799.944,1771.896,1783.296,108247,200,0,XAUUSDm
3,2021-10-31 00:00:00+00:00,1783.269,1783.269,1779.055,1780.026,3100,200,0,XAUUSDm
4,2021-11-01 00:00:00+00:00,1780.020,1795.750,1779.988,1792.047,77556,200,0,XAUUSDm
...,...,...,...,...,...,...,...,...,...
1544,2026-10-14 00:00:00+00:00,4155.136,4170.519,4073.195,4111.850,316484,240,0,XAUUSDm
1545,2026-10-15 00:00:00+00:00,4111.885,4177.440,4111.083,4156.128,269371,240,0,XAUUSDm
1546,2026-10-16 00:00:00+00:00,4155.935,4198.812,4147.115,4165.147,237118,240,0,XAUUSDm
1547,2026-10-18 00:00:00+00:00,4166.610,4185.963,4163.715,4184.701,11291,240,0,XAUUSDm


In [36]:
wti

,time,open,high,low,close,tick_volume,spread,real_volume,symbol
0,2021-10-27 00:00:00+00:00,82.303,82.303,81.959,82.084,613,86,0,USOILm
1,2021-10-28 00:00:00+00:00,81.506,82.608,80.263,82.086,21695,86,0,USOILm
2,2021-10-29 00:00:00+00:00,82.072,82.867,80.797,82.511,30606,86,0,USOILm
3,2021-10-31 00:00:00+00:00,82.197,82.306,81.871,82.094,1140,86,0,USOILm
4,2021-11-01 00:00:00+00:00,82.098,83.928,81.914,82.985,26539,86,0,USOILm
...,...,...,...,...,...,...,...,...,...
1544,2026-10-14 00:00:00+00:00,93.762,94.922,93.102,93.606,167289,20,0,USOILm
1545,2026-10-15 00:00:00+00:00,93.599,96.720,92.455,96.079,132759,20,0,USOILm
1546,2026-10-16 00:00:00+00:00,96.076,99.269,95.215,98.873,116778,20,0,USOILm
1547,2026-10-18 00:00:00+00:00,98.508,98.634,97.543,97.553,2508,20,0,USOILm


In [37]:
gbpusd

,time,open,high,low,close,tick_volume,spread,real_volume,symbol
0,2014-01-14 00:00:00+00:00,1.63878,1.64633,1.63669,1.64318,53646,0,0,GBPUSDm
1,2014-01-15 00:00:00+00:00,1.64325,1.64369,1.63222,1.63601,63863,0,0,GBPUSDm
2,2014-01-16 00:00:00+00:00,1.63612,1.63815,1.63124,1.63458,57297,0,0,GBPUSDm
3,2014-01-17 00:00:00+00:00,1.63445,1.64576,1.63088,1.64170,57879,0,0,GBPUSDm
4,2014-01-19 00:00:00+00:00,1.64167,1.64197,1.64091,1.64165,1954,0,0,GBPUSDm
...,...,...,...,...,...,...,...,...,...
3937,2026-10-14 00:00:00+00:00,1.32565,1.32751,1.31364,1.32289,51418,4,0,GBPUSDm
3938,2026-10-15 00:00:00+00:00,1.32287,1.32780,1.32208,1.32700,58362,7,0,GBPUSDm
3939,2026-10-16 00:00:00+00:00,1.32698,1.33714,1.32673,1.33275,42656,10,0,GBPUSDm
3940,2026-10-18 00:00:00+00:00,1.33028,1.33360,1.32960,1.33353,2414,10,0,GBPUSDm


In [38]:
import matplotlib.pyplot as plt 
from matplotlib import rcParams
rcParams['figure.figsize'] = 15,6
import ta

# Integrating macro data with their respective forex market counterparts

In [39]:
def join_macro_forex_indicators(df_forex, **df_macro):
    
    from ta.trend import EMAIndicator, ADXIndicator
    from ta.volatility import AverageTrueRange
    from ta.volatility import BollingerBands
    from ta.volatility import KeltnerChannel
    from ta.volatility import DonchianChannel

    # Prepare Forex data
    df = df_forex.copy()
    df["time"] = pd.to_datetime(df["time"], utc=True)
    df = df.sort_values("time").drop_duplicates("time", keep="last")
    df["year_month"] = df["time"].dt.tz_localize(None).dt.to_period("M")

    # Join all macroeconomic datasets
    for country, df_country in df_macro.items():
        macro = df_country.copy()

        macro["reference_date"] = pd.to_datetime(
            macro["reference_date"], utc=True
        ).dt.tz_localize(None)

        columns = [
            "inflation_yoy",
            "central_bank_rate",
            "real_gdp_growth_qoq"
        ]

        macro = macro.sort_values("reference_date")
        macro["year_month"] = macro["reference_date"].dt.to_period("M")
        macro = macro.drop_duplicates("year_month", keep="last")

        macro = macro[["year_month"] + columns].rename(
            columns={c: f"{country}_{c}" for c in columns}
        )

        df = df.merge(
            macro,
            on="year_month",
            how="left",
            validate="many_to_one"
        )

    # Fill missing macroeconomic values after all joins
    macro_columns = [
        c for c in df.columns
        if c.endswith((
            "inflation_yoy",
            "central_bank_rate",
            "real_gdp_growth_qoq"
        ))
    ]
    df[macro_columns] = df[macro_columns].ffill()

    # Remove temporary column and set time index
    df = df.drop(columns="year_month")
    df = df.set_index("time").sort_index()

    # Calculate technical indicators
    df["daily_return"] = df["close"].pct_change() * 100
    df["daily_range"] = df["high"] - df["low"]
    df["daily_volatility"] = df["close"].pct_change().rolling(5).std() * 100

    df["ema21"] = EMAIndicator(
        close=df["close"], window=21, fillna=False
    ).ema_indicator()

    df["ema55"] = EMAIndicator(
        close=df["close"], window=55, fillna=False
    ).ema_indicator()

    df["adx14"] = ADXIndicator(
        high=df["high"],
        low=df["low"],
        close=df["close"],
        window=14,
        fillna=False
    ).adx()

    df["atr14"] = AverageTrueRange(
        high=df["high"],
        low=df["low"],
        close=df["close"],
        window=14,
        fillna=False
    ).average_true_range()

    # Calculate Bollinger Bands
    bb = BollingerBands(
        close=df['close'],
        window=20,
        window_dev=2
    )

    # Calculate Bollinger Band Width (%)
    df['bbw'] = (
        (bb.bollinger_hband() - bb.bollinger_lband())
        / bb.bollinger_mavg()
    ) * 100

    # Calculate logarithmic returns
    df['log_return'] = np.log(df['close'] / df['close'].shift(1))

    # Calculate 20-period historical volatility
    df['vh'] = df['log_return'].rolling(window=20).std()

    # Annualize volatility for daily candles
    df['vh_anualizada'] = df['vh'] * np.sqrt(252) * 100

    # Calculate the Keltner Channels
    kc = KeltnerChannel(
        high=df['high'],
        low=df['low'],
        close=df['close'],
        window=20,
        window_atr=10,
        multiplier=2,
        fillna=False
    )

    # Upper, middle, and lower bands
    df['kc_upper'] = kc.keltner_channel_hband()
    df['kc_middle'] = kc.keltner_channel_mband()
    df['kc_lower'] = kc.keltner_channel_lband()

    # Calculate the Normalized Average True Range (NATR)
    df['natr'] = (df['atr14'] / df['close']) * 100

    # Calculate the Donchian Channel
    donchian = DonchianChannel(
        high=df['high'],
        low=df['low'],
        close=df['close'],
        window=20,
        offset=0,
        fillna=False
    )

    # Calculate the Donchian Channel Width
    df['donchian_width'] = (
        (donchian.donchian_channel_hband() -
        donchian.donchian_channel_lband())
        / df['close']
    ) * 100

    # Calculate Garman-Klass Volatility
    df['garman_klass_volatility'] = np.sqrt(
        (
            0.5 * np.log(df['high'] / df['low']) ** 2
            - (2 * np.log(2) - 1)
            * np.log(df['close'] / df['open']) ** 2
        ).clip(lower=0)
    )

    # Remove rows with missing values after calculating indicators
    df = df.dropna()

    # Drop unnecessary columns if present
    cols_drop = ["tick_volume", "spread", "real_volume", "symbol"]
    df = df.drop(columns=cols_drop, errors="ignore")

    return df

In [40]:
eurcad = join_macro_forex_indicators(eurcad, euro=euro_zone, canada=canada)

In [41]:
eurcad

,open,high,low,close,euro_inflation_yoy,euro_central_bank_rate,euro_real_gdp_growth_qoq,canada_inflation_yoy,canada_central_bank_rate,canada_real_gdp_growth_qoq,...,bbw,log_return,vh,vh_anualizada,kc_upper,kc_middle,kc_lower,natr,donchian_width,garman_klass_volatility
time,,,,,,,,,,,,,,,,,,,,,
2021-12-29 00:00:00+00:00,1.44974,1.45550,1.44547,1.45198,1.944361,2.0,0.436587,4.803493,0.25,1.790402,...,2.242883,0.001585,0.003619,5.745124,1.455439,1.446540,1.437641,0.600193,3.000730,0.004795
2021-12-30 00:00:00+00:00,1.45205,1.45218,1.44159,1.44302,1.944361,2.0,0.436587,4.803493,0.25,1.790402,...,1.806564,-0.006190,0.003080,4.889663,1.455826,1.447372,1.438918,0.613202,2.927194,0.003428
2021-12-31 00:00:00+00:00,1.44298,1.44384,1.43489,1.43906,1.944361,2.0,0.436587,4.803493,0.25,1.790402,...,1.717302,-0.002748,0.002793,4.433611,1.455939,1.447763,1.439588,0.615393,2.364043,0.004059
2022-01-02 00:00:00+00:00,1.43531,1.43966,1.43531,1.43966,1.944361,2.0,0.436587,5.137482,0.25,0.636564,...,1.600457,0.000417,0.002793,4.433001,1.456107,1.447944,1.439781,0.592780,2.363058,0.001020
2022-01-03 00:00:00+00:00,1.43969,1.44389,1.43602,1.44110,1.944361,2.0,0.436587,5.137482,0.25,0.636564,...,1.575961,0.001000,0.002699,4.283885,1.456188,1.448119,1.440050,0.588897,2.056068,0.003816
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-10-14 00:00:00+00:00,1.59238,1.59250,1.58612,1.59086,1.944361,2.0,0.436587,1.992582,2.25,0.636564,...,1.731698,-0.000974,0.002397,3.804621,1.608567,1.602451,1.596335,0.366070,1.738682,0.002776
2026-10-15 00:00:00+00:00,1.59084,1.59608,1.58933,1.59141,1.944361,2.0,0.436587,1.992582,2.25,0.636564,...,1.770033,0.000346,0.002381,3.779497,1.607844,1.601619,1.595394,0.370101,1.738081,0.002988
2026-10-16 00:00:00+00:00,1.59140,1.59456,1.58791,1.58886,1.944361,2.0,0.436587,1.992582,2.25,0.636564,...,1.883783,-0.001604,0.002315,3.674268,1.607034,1.600829,1.594624,0.374113,1.740871,0.002783


In [42]:
eurchf = join_macro_forex_indicators(eurchf, euro=euro_zone, suice=switz)
eurgbp = join_macro_forex_indicators(eurgbp, euro=euro_zone, england=england)
eurjpy = join_macro_forex_indicators(eurjpy, euro=euro_zone, japan=japan)
eurusd = join_macro_forex_indicators(eurusd, euro=euro_zone, usa=usa)
usdcad = join_macro_forex_indicators(usdcad, usa=usa, canada=canada)
usdchf = join_macro_forex_indicators(usdchf, usa=usa, suice=switz)
usdjpy = join_macro_forex_indicators(usdjpy, usa=usa, japan=japan)
gold = join_macro_forex_indicators(gold, usa=usa, suice=switz)
silver = join_macro_forex_indicators(silver, usa=usa, suice=switz)
brent = join_macro_forex_indicators(brent, usa=usa, canada=canada)
wti = join_macro_forex_indicators(wti, usa=usa, canada=canada)

KeyError: "['inflation_yoy', 'central_bank_rate', 'real_gdp_growth_qoq'] not in index"

In [ ]:
eurcad = eurcad.round(5)
eurchf = eurchf.round(5)
eurgbp = eurgbp.round(5)
eurjpy = eurjpy.round(5)
eurusd = eurusd.round(5)
usdcad = usdcad.round(5)
usdchf = usdchf.round(5)
usdjpy = usdjpy.round(5)
gold = gold.round(5)
silver = silver.round(5)
brent = brent.round(5)
wti = wti.round(5)

In [ ]:
eurcad

In [ ]:
brent

In [ ]:
usdcad

In [ ]:
eurusd

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
def split_base_model(df, target='close', test_size=0.025):

    from sklearn.model_selection import train_test_split

    # Copy df
    df = df.copy().sort_index()

    # Dependent variable and exogens variable
    y = df[target]
    X = df.drop(columns=[target])

    # It separates the data up to the previous period, preventing data leaks and model bias.
    X = X.shift(1)

    data = pd.concat(
        [y.rename(target), X], axis=1
    ).dropna()

    # Split dataframe in train and test
    split = int(len(data) * (1 - test_size))

    X_train = data.drop(columns=[target]).iloc[:split]
    X_test = data.drop(columns=[target]).iloc[split:]

    y_train = data[target].iloc[:split]
    y_test = data[target].iloc[split:]
    
    # Return df
    return X_train, X_test, y_train, y_test

In [ ]:
X_train_basemodel, X_test_basemodel, y_train_basemodel, y_test_basemodel = split_base_model(eurusd)

In [ ]:
base_model = SARIMAX(y_train_basemodel, exog=X_train_basemodel, order=(1,0,1), seasonal_order=(1,1,1,7))

In [ ]:
base_model_fit = base_model.fit()

In [ ]:
print(base_model_fit.summary())

In [ ]:
base_model_forecast = base_model_fit.forecast(steps=len(y_test_basemodel), exog=X_test_basemodel)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# Actual and predicted values
actual_basemodel = np.asarray(y_test_basemodel).ravel()
predicted_basemodel = np.asarray(base_model_forecast).ravel()

# MAE: Mean Absolute Error
mae_basemodel = mean_absolute_error(actual_basemodel, predicted_basemodel)

# RMSE: Root Mean Squared Error
rmse_basemodel = np.sqrt(mean_squared_error(actual_basemodel, predicted_basemodel))

# MAPE: Mean Absolute Percentage Error
mape_basemodel = np.mean(np.abs((actual_basemodel - predicted_basemodel) / actual_basemodel)) * 100

print(f"MAE: {mae_basemodel:.6f}")
print(f"RMSE: {rmse_basemodel:.6f}")
print(f"MAPE: {mape_basemodel:.4f}%")

In [ ]:
def compare_forecast_real(y_test, y_pred):
    """
    Combine actual and predicted values into a DataFrame
    and calculate forecast errors.
    """
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    import numpy as np

    # Convert inputs to one-dimensional arrays
    actual_values = np.asarray(y_test).ravel()
    predicted_values = np.asarray(y_pred).ravel()

    # Ensure boths inputs contain the same number of observations
    if len(actual_values) != len(predicted_values):
        raise ValueError('Actual and predicted values must have same length')
    
    # Preserve the actual values index when available
    if isinstance(y_test, (pd.Series, pd.DataFrame)):
        index = y_test.index
    else:
        index = pd.RangeIndex(start=0, stop=len(actual_values))

    # Create compare DataFrame
    compare_df = pd.DataFrame({
        'actual': actual_values,
        'predicted': predicted_values
    }, index=index)

    # Calculated signed and absolute forecast errors
    compare_df['error'] = (compare_df['predicted'] - compare_df['actual'])

    compare_df['absolute_error'] = (compare_df['error'].abs())

    compare_df['mae'] = mean_absolute_error(y_test, y_pred)

    compare_df['rmse'] = np.sqrt(mean_squared_error(y_test, y_pred))
    
    compare_df['mse'] = mean_squared_error(y_test, y_pred)
 
    # Return df
    return compare_df

In [ ]:
compare_df_base_model = compare_forecast_real(y_test_basemodel, base_model_forecast)

In [ ]:
compare_df_base_model = compare_df_base_model.round(5)

In [ ]:
def compare_forecast_plot(compare_df, title='Actual market x Predicted values'):
    """
    Plot actual and predicted prices over time.
    """

    # Sort observations chronologically when using a time index
    plot_df = compare_df.sort_index()

    #Plot actual and predicted values
    plt.plot(
        plot_df.index,
        plot_df['actual'],
        label='Actual Price',
        linewidth = 0.88
    )

    plt.plot(
        plot_df.index,
        plot_df['predicted'],
        label='Predicted Price',
        linewidth=0.65,
        linestyle='--'
    )

    # Add chart labels and formatting
    plt.title(title)
    plt.xlabel('Time')
    plt.ylabel('Price')
    plt.legend()
    plt.grid(True)

    # Improve date label formatting
    plt.tight_layout()
    plt.show()

In [ ]:
compare_forecast_plot(compare_df_base_model)

In [ ]:
compare_df_base_model

In [ ]:
base_model_summary = base_model_fit.summary()

In [ ]:
base_model_summary

In [ ]:
def split2_sarimax(df, target='close', test_size=0.025):

    # Df copy
    df = df.copy().sort_index()

    # Create a columns indicators to exogen variable
    cols_indicators = [
        'ema21','ema55','low','high','open','daily_range','euro_inflation_yoy',
        'euro_central_bank_rate','usa_inflation_yoy','usa_central_bank_rate','atr14',
        'garman_klass_volatility','natr','kc_lower','kc_middle','kc_upper','vh',
        'log_return','daily_volatility','daily_return'
    ]

    features = cols_indicators

    y = df[target]
    X = df[features].shift(1)

    # Merge y and X to dataframe
    data = pd.concat(
        [y.rename(target), X], axis=1
    ).dropna()

    # Split dataframe in train and test
    split = int(len(data) * (1 - test_size))

    X_train = data.drop(columns=[target]).iloc[:split]
    X_test = data.drop(columns=[target]).iloc[split:]

    y_train = data[target].iloc[:split]
    y_test = data[target].iloc[split:]
    
    # Return df
    return X_train, X_test, y_train, y_test

In [ ]:
X_train_basemodel2, X_test_basemodel2, y_train_basemodel2, y_test_basemodel2 = split2_sarimax(eurusd)

In [ ]:
X_train_basemodel.shape, X_test_basemodel.shape, y_train_basemodel.shape, y_test_basemodel.shape

In [ ]:
X_train_basemodel2.shape, X_test_basemodel2.shape, y_train_basemodel2.shape, y_test_basemodel2.shape

In [ ]:
base_model2 = SARIMAX(y_train_basemodel2, exog=X_train_basemodel2, order=(1,0,1), seasonal_order=(1,1,1,7))

In [ ]:
base_model2_fit = base_model2.fit()

In [ ]:
base_model2_summary = base_model2_fit.summary()

In [ ]:
base_model2_summary

In [ ]:
forecast_basemodel2 = base_model2_fit.forecast(steps=len(y_test_basemodel2), exog=X_test_basemodel2)

In [ ]:
compare_basemodel2 = compare_forecast_real(y_test_basemodel2, forecast_basemodel2)

In [ ]:
compare_basemodel2 = compare_basemodel2.round(5)

In [ ]:
compare_forecast_plot(compare_basemodel2)

In [ ]:
compare_basemodel2

In [ ]:
eurusd

This project incorporates basic macroeconomic data such as GDP, inflation, and interest rates.

I applied the development of some technical market indicators, such as:

- Exponential Moving Averages
- ADX indicator (measures trend strength)
- ATR (measures the strength of the volatility of the asset being traded over a period X of time)
- Bollinger Bands (Measures the standard deviation around the average over X periods of time)
- Keltner Channel (Similar to Bollinger Bands, but uses ATR to measure the distance of prices from the average)
- Donchian Channel Width (Amplitude between the minimum and maximum of a channel)
- Garman-Klass Volatility (Volatility estimated by high, low, close, and open)
- NATR (Expresses ATR as a percentage of the price. Useful for describing the volatility of relative pairs)
- Historical Volatility (Historical price volatility over X periods of days)
- Annualized Historical Volatility (Same principle, but relative to the year)

Here, data relationships were also developed between Macroeconomic indicators for each country and their respective trading partners, along with an analysis of technical indicators.

Two basic econometric models, named SARIMAX, were developed: model 1 and model 2, respectively. Model 1 was run using all indicators to predict the closing price, and the final error value (subtraction of prediction from actual price) proved to be very small, almost negligible in the trade if the analysis was incorrect.

Model 2, however, involved streamlining the indicators applied to the model's learning process. It also showed a very small error compared to that practiced in the real market, with only a few error lines, even represented in scientific notation due to the small size of the error.

Both models are capturing daily volatility very well, and the way the econometric models capture price trends is extremely satisfactory, presenting very satisfactory results so far.

I intend to continue attribute engineering, eventually developing a final model that will run only with what is necessary and most important for its learning and prediction. To date, the decision to apply inflation, interest rates, and indicators that measure the asset's trend and volatility remains the best decision. I hope this summary has made the method applied so far clear and without a doubt. Even with the order of (1,0,1), which is what we call a random walk, and the seasonal order of (1,1,1,7), representing respectively autoregression, differentiation, and seasonal moving average applied to the 7-day seasonal period, it has proven very efficient.

I also intend to run a model now to find the best parameter for p,d,q and P,D,Q,S to identify if there are areas for automated improvement.

Below is a graph demonstrating the predictive capacity of the two models in relation to the actual market price.

In [ ]:
plt.plot(compare_df_base_model.index, compare_df_base_model.predicted, color='blue', linewidth=0.85, linestyle='--', label='Model 1')
plt.plot(compare_basemodel2.index, compare_basemodel2.predicted, color='red', linewidth=0.85, linestyle='--', label='Model 2')
plt.plot(compare_basemodel2.index, compare_basemodel2.actual, color='green', linewidth=0.95, label='Real price')
plt.legend()
plt.xlabel('Time')
plt.ylabel('Price')
plt.title('Model 1 and Model 2 predict values x Real price Market')
plt.grid(True)
plt.show()